# Tutorial 14: Expressions Come Alive

**Programming Design Principles / Maths for IT**

Today we take on one of the most satisfying challenges in these tutorials: representing algebraic expressions as data structures and computing with them. A polynomial like $3x^2 + 5x - 2$ will become a list of numbers that our functions can evaluate, add, and multiply. The algebra becomes tangible.

## Expressions versus Equations

An important distinction first. An *expression* is a mathematical phrase that has a value: $3x + 7$, $x^2 - 4$, $\frac{x+1}{x-1}$. It does not assert anything -- it just computes something for a given value of x.

An *equation* is a statement that two expressions are equal: $3x + 7 = 22$, $x^2 - 4 = 0$. An equation asserts something and can be true or false depending on x.

We *evaluate* expressions. We *solve* equations. Today is about evaluation; solving comes in Tutorial 15.

## Representing Polynomials

A polynomial like $3x^2 + 5x - 2$ has a simple structure: it is a sum of terms, each being a coefficient multiplied by a power of x. We can represent it as a list of coefficients, where the position in the list indicates the power.

We will use the convention that index $i$ holds the coefficient of $x^i$. So:

$$3x^2 + 5x - 2 \quad\leftrightarrow\quad [-2, 5, 3]$$

The constant term ($-2$, coefficient of $x^0$) is at index 0, the coefficient of $x^1$ (which is 5) is at index 1, and the coefficient of $x^2$ (which is 3) is at index 2.

This convention is natural because the index matches the exponent.

In [1]:
# Some polynomials as lists
constant_5 = [5]               # just the number 5
linear = [3, 2]                # 2x + 3
quadratic = [-2, 5, 3]        # 3x^2 + 5x - 2
cubic = [1, 0, -3, 2]         # 2x^3 - 3x^2 + 1

print("Constant:", constant_5)
print("Linear:", linear)
print("Quadratic:", quadratic)
print("Cubic:", cubic)

Constant: [5]
Linear: [3, 2]
Quadratic: [-2, 5, 3]
Cubic: [1, 0, -3, 2]


## Evaluating Polynomials

To evaluate $3x^2 + 5x - 2$ at $x = 4$, we compute: $3(16) + 5(4) - 2 = 48 + 20 - 2 = 66$.

In terms of our list: for each index $i$, multiply the coefficient by $x^i$ and add them all up.

$$p(x) = \sum_{i=0}^{n} c_i \cdot x^i$$

This is a direct application of sigma notation -- and it maps perfectly to a loop.

### Your turn

Write a function `evaluate_poly(coeffs, x)` that takes a list of coefficients and a value of x, and returns the polynomial's value at that point.

**Pseudocode:**
```
SET result = 0
FOR each index i from 0 to length-1:
    ADD coeffs[i] * x^i to result
RETURN result
```

In [2]:
def evaluate_poly(coeffs, x):
    """Return the value of the polynomial represented by coeffs at the given x.

    coeffs[i] is the coefficient of x^i.
    """
    result = 0
    for i in range(len(coeffs)):
        result = result + coeffs[i] * x ** i
    return result


In [3]:
print(evaluate_poly([-2, 5, 3], 0))
print(evaluate_poly([-2, 5, 3], 1))
print(evaluate_poly([-2, 5, 3], 4))
print(evaluate_poly([1], 999))


-2
6
66
1


## Displaying Polynomials

A list like `[-2, 5, 3]` is fine for computation but not great for reading. Let's write a function that produces a human-readable string like `"3x^2 + 5x - 2"`.

This is trickier than it looks. We need to handle:
- Zero coefficients (skip them)
- The constant term (no "x" part)
- The linear term ($x^1$ should display as just "x", not "x^1")
- Coefficient of 1 or -1 (display as "x^2" not "1x^2")
- Positive and negative signs (the first term should not start with "+")

### Your turn

Write a function `poly_to_string(coeffs)` that returns a human-readable string. Start with a simple version that works for basic cases, then refine it to handle the edge cases above.

Do not worry about making it perfect on the first try -- string formatting with many special cases is genuinely tricky. Get the basic version working first, then improve.

In [4]:
def poly_to_string(coeffs):
    """Return a human-readable string like "3x^2 + 5x - 2" for the given coefficient list.

    We build the string from the highest power down to the constant term,
    skipping zero coefficients and handling the x^0 and x^1 cases specially.
    """
    terms = []
    degree = len(coeffs) - 1
    for i in range(degree, -1, -1):
        c = coeffs[i]
        if c == 0:
            continue

        if i == 0:
            term = str(abs(c))
        elif i == 1:
            if abs(c) == 1:
                term = "x"
            else:
                term = str(abs(c)) + "x"
        else:
            if abs(c) == 1:
                term = "x^" + str(i)
            else:
                term = str(abs(c)) + "x^" + str(i)

        if not terms:
            # First term written: keep the sign only if negative
            if c < 0:
                terms.append("-" + term)
            else:
                terms.append(term)
        else:
            if c < 0:
                terms.append("- " + term)
            else:
                terms.append("+ " + term)

    if not terms:
        return "0"
    return " ".join(terms)


In [5]:
print(poly_to_string([-2, 5, 3]))
print(poly_to_string([0, 0, 1]))
print(poly_to_string([7]))
print(poly_to_string([0, 1]))


3x^2 + 5x - 2
x^2
7
x


## Adding Polynomials

Adding two polynomials means adding corresponding coefficients:

$$(3x^2 + 5x - 2) + (x^2 - 3x + 7) = 4x^2 + 2x + 5$$

In list form: `[-2, 5, 3] + [7, -3, 1] = [5, 2, 4]`

When the polynomials have different degrees (different list lengths), the shorter one effectively has zeros in the higher positions.

### Your turn

Write a function `add_poly(a, b)` that returns a new list representing the sum. Handle different-length lists gracefully.

**Pseudocode:**
```
SET length to the longer of the two lists
CREATE result list of that length, filled with zeros
FOR each index i in result:
    IF i < length of a: ADD a[i] to result[i]
    IF i < length of b: ADD b[i] to result[i]
RETURN result
```

In [6]:
def add_poly(a, b):
    """Return a new coefficient list representing a + b."""
    length = max(len(a), len(b))
    result = [0] * length
    for i in range(length):
        if i < len(a):
            result[i] = result[i] + a[i]
        if i < len(b):
            result[i] = result[i] + b[i]
    return result


In [7]:
print(add_poly([-2, 5, 3], [7, -3, 1]))
print(add_poly([1, 1], [1, 1, 1, 1]))


[5, 2, 4]
[2, 2, 1, 1]


## Multiplying Polynomials

Multiplying polynomials is more involved. When we multiply $(2x + 3)(x + 4)$, we use the FOIL method (or more generally, distribute each term of the first polynomial across every term of the second):

$$(2x + 3)(x + 4) = 2x^2 + 8x + 3x + 12 = 2x^2 + 11x + 12$$

The key insight: when we multiply $c_i x^i$ by $c_j x^j$, the result is $c_i \cdot c_j \cdot x^{i+j}$. So the coefficient at position $k$ in the result is the sum of all products $a_i \cdot b_j$ where $i + j = k$.

### Your turn

Write a function `multiply_poly(a, b)` that returns a new list representing the product.

**Pseudocode:**
```
SET result_length = length(a) + length(b) - 1
CREATE result list of that length, filled with zeros
FOR each index i in a:
    FOR each index j in b:
        ADD a[i] * b[j] to result[i + j]
RETURN result
```

In [8]:
def multiply_poly(a, b):
    """Return a new coefficient list representing the product a * b."""
    result_length = len(a) + len(b) - 1
    result = [0] * result_length
    for i in range(len(a)):
        for j in range(len(b)):
            result[i + j] = result[i + j] + a[i] * b[j]
    return result


In [9]:
print(multiply_poly([3, 2], [4, 1]))
print(multiply_poly([1, 1], [1, 1]))


[12, 11, 2]
[1, 2, 1]


### A verification trick

We can verify polynomial multiplication by evaluating both sides at a specific value of x. If $(2x + 3)(x + 4) = 2x^2 + 11x + 12$, then both sides should give the same value for any x:

In [10]:
# Verification by evaluation
a = [3, 2]     # 2x + 3
b = [4, 1]     # x + 4
product = multiply_poly(a, b)

x = 5
left_side = evaluate_poly(a, x) * evaluate_poly(b, x)
right_side = evaluate_poly(product, x)
print("(2*5+3) * (5+4) =", left_side)
print("2*25 + 11*5 + 12 =", right_side)
print("Match:", left_side == right_side)

(2*5+3) * (5+4) = 117
2*25 + 11*5 + 12 = 117
Match: True


This is a powerful testing technique: use a known mathematical property to verify your code. If `evaluate_poly(multiply_poly(a, b), x)` equals `evaluate_poly(a, x) * evaluate_poly(b, x)` for several values of x, your multiplication is almost certainly correct.

### Your turn

Write a function `test_multiply(a, b)` that automatically verifies the multiplication by checking the evaluation at x = 0, 1, 2, -1, and 10. Print PASS or FAIL for each.

In [11]:
def test_multiply(a, b):
    """Verify multiply_poly(a, b) by checking evaluate_poly at several x values."""
    product = multiply_poly(a, b)
    for x in [0, 1, 2, -1, 10]:
        left_side = evaluate_poly(a, x) * evaluate_poly(b, x)
        right_side = evaluate_poly(product, x)
        if abs(left_side - right_side) < 1e-9:
            print("x =", x, "PASS")
        else:
            print("x =", x, "FAIL", left_side, "!=", right_side)


In [12]:
test_multiply([3, 2], [4, 1])
test_multiply([1, 1], [1, 1])
test_multiply([-2, 5, 3], [7, -3, 1])


x = 0 PASS
x = 1 PASS
x = 2 PASS
x = -1 PASS
x = 10 PASS
x = 0 PASS
x = 1 PASS
x = 2 PASS
x = -1 PASS
x = 10 PASS
x = 0 PASS
x = 1 PASS
x = 2 PASS
x = -1 PASS
x = 10 PASS


## Subtracting and Scaling

Two more operations before we finish: subtracting polynomials and multiplying by a constant (scaling).

### Your turn

Write `subtract_poly(a, b)` and `scale_poly(coeffs, scalar)`. Think about how each relates to what you have already built.

In [13]:
def subtract_poly(a, b):
    """Return a new coefficient list representing a - b.

    This is addition of a and the negation of b, so we can lean on scale_poly and add_poly.
    """
    return add_poly(a, scale_poly(b, -1))


def scale_poly(coeffs, scalar):
    """Return a new coefficient list with every coefficient multiplied by scalar."""
    return [c * scalar for c in coeffs]


In [14]:
print(subtract_poly([-2, 5, 3], [7, -3, 1]))
print(scale_poly([-2, 5, 3], 2))


[-9, 8, 2]
[-4, 10, 6]


## Reflection

We have built the core of a polynomial algebra system: representation, evaluation, display, addition, subtraction, multiplication, and scaling. Each function is small, testable, and builds on the others.

The deeper lesson is about *representation*. By choosing to represent polynomials as lists, we turned abstract algebra into concrete data manipulation. Addition became adding list elements. Multiplication became a nested loop. The algebra did not change -- our perspective on it did.

Next time we will use this machinery to solve equations and factor polynomials.

What was the trickiest part of this tutorial? The `poly_to_string` formatting, or the `multiply_poly` algorithm?

